# Lab 2 — ReVA VQA: Qwen3-VL Fine-Tuning and VILA Comparison

**Course**: CSE 498
**Assignment repository**: https://github.com/likaiw2/VLM_evaluating_finetuing
**My code (GitHub)**: https://github.com/shinnew9/CSE498_AI-Healthcare-Robotics/tree/main/Lab2_ReVA_VQA

---

## 0. Summary

Three models were evaluated on the ReVA benchmark (aerial / drone video QA) under identical conditions.

| Model | Accuracy | Completion |
|---|---|---|
| Qwen3-VL-4B-Instruct (base) | **70.44%** (143/203) | 100.00% |
| Qwen3-VL-4B-Instruct (LoRA fine-tuned) | **70.94%** (144/203) | 100.00% |
| VILA1.5-3B (base) | **55.17%** (112/203) | 100.00% |

The evaluation set is a **50-video / 203-question** subset drawn from the official ReVA `test_set.json`.
All three models were evaluated on exactly the same questions.

**Key findings**

1. Qwen3-VL-4B outperforms VILA1.5-3B by **15.3 percentage points** overall.
2. The largest gap is in **Temporal Grounding** (58.8% vs 17.6%); VILA scores below random chance (25%) there.
3. **Perspective and Viewpoint** is the only subcategory where Qwen loses to VILA.
4. Fine-tuning changed overall accuracy by only **+0.50 pp (1 question)**; the learning rate (`2e-7`) was too small for the loss to move.

## 1. Commands Executed

All commands were run from the project root (`~/venvs/CSE498_ProfC/VLM_evaluating_finetuing`).

### 1.0 Environment setup

```bash
# Register conda on PATH (needed in every new shell)
export PATH=/opt/tljh/user/bin:$PATH

# Environment for Qwen evaluation / training
conda create -n qwen2 python=3.10 -y
conda run -n qwen2 pip install torch==2.6.0
conda run -n qwen2 pip install transformers accelerate decord tqdm numpy qwen-vl-utils
conda run -n qwen2 pip install opencv-python-headless torchvision==0.21.0
conda run -n qwen2 pip install peft deepspeed

# Environment for VILA (official setup script)
git clone https://github.com/NVlabs/VILA.git
cd VILA && bash environment_setup.sh vila

# ffmpeg was not installed on the server; use the pip static build
pip install imageio-ffmpeg
ln -sf $(python3 -c "import imageio_ffmpeg;print(imageio_ffmpeg.get_ffmpeg_exe())") ~/.local/bin/ffmpeg
export PATH="$HOME/.local/bin:$PATH"
```

### 1.1 Downloading data and model weights

```bash
hf auth login --token <TOKEN>

hf download ReVA-Benchmark/ReVA --repo-type dataset \
  --local-dir ~/venvs/CSE498_ProfC/datasets/ReVA          # 29.9 GB, 1273 files

hf download Qwen/Qwen3-VL-4B-Instruct \
  --local-dir ~/venvs/CSE498_ProfC/models/Qwen3-VL-4B-Instruct

hf download Efficient-Large-Model/VILA1.5-3b \
  --local-dir ~/venvs/CSE498_ProfC/models/VILA1.5-3b
```

### 1.2 Part 1 — Converting ReVA annotations into Qwen training data

```bash
# Verify the pipeline on the bundled demo data first
python3 scripts/setup_demo_data.py
python3 scripts/check_student_setup.py

# Link the real videos: the training data loader resolves paths
# relative to data/qwen_train
cd data/qwen_train
for d in VisDrone ERA_Select Hawk_UAV UAVDT; do
  ln -sfn ~/venvs/CSE498_ProfC/datasets/ReVA/$d $d
done
cd ../..

# Build the real 200-sample training set
python3 scripts/build_qwen_train_data.py \
  --input data/reva_train/train_set.json \
  --output data/qwen_train/train.json \
  --video-root data/qwen_train \
  --max-samples 200 \
  --answer-style tagged --prompt-style reva_eval \
  --require-video --keep-metadata

# For reference: converting the full split yields 15,773 samples (train_full.json)
```

### 1.3 Building the evaluation subset (50 videos / 203 questions)

```python
import json
d = json.load(open('~/venvs/CSE498_ProfC/datasets/ReVA/test_set.json'))
subset = dict(list(d['videos'].items())[:50])
json.dump({'metadata': d.get('metadata', {}), 'videos': subset},
          open('~/venvs/CSE498_ProfC/eval_subset/test_set_50v.json', 'w'),
          ensure_ascii=False)
```

### 1.4 Part 2 — Qwen baseline evaluation

```bash
CONDA_ENV=qwen2 \
EVAL_GPU=0 \
EVAL_BATCH_SIZE=2 \
PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
MODEL_PATH=~/venvs/CSE498_ProfC/models/Qwen3-VL-4B-Instruct \
BACKEND=transformers \
MAX_FRAMES=4 \
REVA_ROOT=~/venvs/CSE498_ProfC/datasets/ReVA \
REVA_JSON=~/venvs/CSE498_ProfC/eval_subset/test_set_50v.json \
EVAL_NAME=qwen_base_real50e \
bash scripts/run_eval_qwen_base.sh
```

Outputs: `outputs/qwen_base_real50e/qwen_base_real50e/{result.csv, result.json, pred.json}`

### 1.5 Part 3 — VILA baseline evaluation

```bash
CONDA_ENV=vila \
VILA_REPO=~/venvs/CSE498_ProfC/repos/VILA \
MODEL_PATH=~/venvs/CSE498_ProfC/models/VILA1.5-3b \
NUM_VIDEO_FRAMES=4 \
GPU=3 \
REVA_ROOT=~/venvs/CSE498_ProfC/datasets/ReVA \
REVA_JSON=~/venvs/CSE498_ProfC/eval_subset/test_set_50v.json \
OUTPUT_DIR=outputs/vila_reva_v2_real50 \
bash scripts/run_eval_reva_vila.sh
```

Outputs: `outputs/vila_reva_v2_real50/{metrics.json, outputs.jsonl}`

### 1.6 Part 4 — Qwen LoRA fine-tuning

A single 11 GB GPU could not hold the model plus training state, so DeepSpeed ZeRO-3 with
**CPU parameter offload** was used (see §5.4).

```bash
CONDA_ENV=qwen2 \
CUDA_VISIBLE_DEVICES=3 \
DEEPSPEED_CONFIG=./scripts/zero3_offload.json \
PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
MODEL_PATH=~/venvs/CSE498_ProfC/models/Qwen3-VL-4B-Instruct \
NPROC_PER_NODE=1 BATCH_SIZE=1 GRAD_ACCUM_STEPS=4 EPOCHS=1 SAVE_STEPS=50 \
MAX_PIXELS=28224 VIDEO_MAX_FRAMES=2 VIDEO_MAX_PIXELS=28224 VIDEO_FPS=1 \
MODEL_MAX_LENGTH=768 USE_DEEPSPEED=1 \
RUN_NAME=qwen_reva_sft_real OUTPUT_DIR=../outputs/qwen_reva_sft_real \
bash scripts/run_finetune_qwen.sh
```

**Training result**: 50 steps (1 epoch over 200 samples), `train_loss = 2.866`, wall time 2,196 s (36 min).
Output: `outputs/qwen_reva_sft_real/checkpoint-50/`

### 1.7 Part 5 — Evaluating the fine-tuned checkpoint

Split across 4 GPUs in parallel chunks (~7x faster than a single GPU).

```bash
CONDA_ENV=qwen2 \
NUM_CHUNKS=4 \
EVAL_BATCH_SIZE=1 \
PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
MODEL_PATH=outputs/qwen_reva_sft_real/checkpoint-50 \
MODEL_BASE=~/venvs/CSE498_ProfC/models/Qwen3-VL-4B-Instruct \
BACKEND=transformers MAX_FRAMES=4 \
REVA_ROOT=~/venvs/CSE498_ProfC/datasets/ReVA \
REVA_JSON=~/venvs/CSE498_ProfC/eval_subset/test_set_50v.json \
EVAL_NAME=qwen_sft_real50_p4 \
bash scripts/run_eval_qwen_finetuned.sh
```

### 1.8 Part 6 — Model comparison

```bash
python3 scripts/compare_model_metrics.py \
  --qwen-base outputs/qwen_base_real50e/qwen_base_real50e/result.csv \
  --qwen-finetuned outputs/qwen_sft_real50_p4/qwen_sft_real50_p4/result.csv \
  --vila outputs/vila_reva_v2_real50/metrics.json \
  --output outputs/model_comparison.csv
```

## 2. Results

| Subcategory | N | Qwen base | Qwen fine-tuned | VILA |
|---|---|---|---|---|
| **Overall** | **203** | **70.44%** | **70.94%** | **55.17%** |
| Causation Reasoning | 14 | 85.71% | 85.71% | 78.57% |
| Change Detection | 29 | 55.17% | 62.07% | 37.93% |
| Consequence Reasoning | 10 | 80.00% | 80.00% | 60.00% |
| General Understanding | 15 | 93.33% | 100.00% | 93.33% |
| Geometric Relation | 11 | 90.91% | 81.82% | 36.36% |
| Hypothetical Reasoning | 13 | 84.62% | 92.31% | 53.85% |
| Object and Land Cover Recognition | 55 | 72.73% | 70.91% | 61.82% |
| Perspective and Viewpoint | 17 | 29.41% | 29.41% | 47.06% |
| Structural Layout | 11 | 72.73% | 72.73% | 72.73% |
| Temporal Grounding | 17 | 58.82% | 58.82% | 17.65% |
| Trend and Pattern | 11 | 81.82% | 72.73% | 54.55% |

### Observations

1. **Qwen3-VL-4B clearly outperforms VILA1.5-3B** (70.44% vs 55.17%). Part of this is the
   parameter count (4B vs 3B), but the gap concentrates in temporal and geometric reasoning,
   which suggests more than a size effect.

2. **Temporal Grounding shows the widest gap**: VILA 17.65% vs
   Qwen 58.82%. VILA scores below the 25% random-chance
   baseline, i.e. with 4 sampled frames it effectively cannot localise time intervals.

3. **Perspective and Viewpoint is the only subcategory where Qwen loses**
   (Qwen 29.41% vs VILA 47.06%).
   Qwen consistently over-estimates the camera elevation angle (see §3, Case 1).

4. **Change Detection is weak for both models.** These questions require comparing frames over time,
   and uniform 4-frame sampling discards intermediate changes.

## 3. Qualitative Analysis — Success and Failure Cases

Each case is labelled with one of `visual perception` / `temporal reasoning` / `option parsing` / `overfitting`.

### Case 1 — `QA-000049` · Perspective and Viewpoint · **visual perception**

> **Q**: What is the camera viewpoint classification in the video?
> A. ground-level (0 degrees) / B. top-down (90 degrees) / C. high angle (>45 degrees) / D. high angle (<45 degrees)
> **Gold**: D

| Model | Prediction | Correct |
|---|---|---|
| Qwen base | C (>45°) | ✗ |
| VILA | B (top-down 90°) | ✗ |

Both models failed, and **they failed in the same direction — over-estimating the camera elevation angle**.
VILA's own rationale states "elevated perspective... less than 45 degrees from the horizontal", yet it
selected B (90°), so its explanation contradicts its choice. This is a **perception** failure: mapping a
continuous physical quantity (angle) onto discrete options.

### Case 2 — `QA-000083` · Change Detection · **visual perception**

> **Q**: How does the movement of the vehicles change throughout the video?
> **Gold**: C (vehicles move from the bottom of the frame to the top)

| Model | Prediction | Correct |
|---|---|---|
| Qwen base | A (top → bottom) | ✗ |
| Qwen fine-tuned | C | ✓ |
| VILA | A (top → bottom) | ✗ |

**Both baselines picked the exact opposite direction.** With only 4 uniformly sampled frames the
inter-frame interval is large, object correspondence breaks down, and direction estimation becomes
close to random. The fine-tuned model answered correctly here, but given the net change of only
1 question overall (§4), this single flip cannot be attributed to learning with confidence.

### Case 3 — `QA-000010` · Temporal Grounding · **temporal reasoning**

> **Q**: How long does the group of people remain visible in the parking lot during the video?
> **Gold**: C (0.0 s – 5.0 s)

| Model | Prediction | Correct |
|---|---|---|
| Qwen base | C | ✓ |
| VILA | D (2.0 s – 5.0 s) | ✗ |

Qwen reached the answer by **enumerating timestamps explicitly** in its generation:
`"At 0.0s ... At 1.0s ... At 5.0s ..."`. VILA, by contrast, wrote that the subject is visible "from the
very start of the video at 0.0s" and then chose an option whose start time is 2.0 s.
This difference in reasoning style — explicit temporal enumeration versus a single holistic judgement —
plausibly explains the 41 pp gap between the two models in Temporal Grounding.

### Case 4 — `QA-000005` · Object and Land Cover Recognition · **visual perception**

> **Q**: How many cars are visible in the parking lot at the 00:00 mark?
> A. 40-50 / B. 10-20 / C. 20-30 / D. 30-40
> **Gold**: B

| Model | Prediction | Correct |
|---|---|---|
| Qwen base | D (30-40) | ✗ |
| VILA | B | ✓ |

One of the few cases where Qwen fails and VILA succeeds. Qwen asserted
`"approximately 30-40 cars"`, **over-counting small objects** in an aerial frame that had been
downsampled to the 50,176-pixel cap. Input resolution directly limits counting accuracy here.

### Case 5 — `QA-000223` · General Understanding · **option parsing**

> **Q**: Which of the following objects exist in this video?
> A. Park benches / B. Lane dividers / C. Traffic cones / D. Telephone booths
> **Gold**: B

Raw output from Qwen base:

```
... Lane dividers are implied by the road markings and the flow of traffic,
but they are not explicitly visible as physical objects.
Therefore, none of the listed objects are clearly present.

<answer>None of the above</answer>
```

The model **invented an option that does not exist**. `extract_letter()` found no valid A–H letter and
returned an empty string, so the item was scored as incorrect.

Across all 203 questions this was the **only** unparsable response; 198 of the remaining responses
carried a well-formed `<answer>` tag. In other words, **the parser behaved robustly and this single
case is an instruction-following failure by the model**, not a parsing bug. The scorer keeps such items
in the denominator, satisfying the requirement that "a missing or unparsable prediction counts as incorrect".

**Notably**, the fine-tuned model answered the same question with a clean `<answer>B</answer>` and scored
it correct. Since every training target uses the `<answer>X</answer>` format, output-format compliance is
the kind of signal that can be picked up even from a very small amount of training — though with a net
change of one question overall this remains suggestive rather than conclusive (§4).

## 4. Fine-Tuning Analysis (overfitting perspective)

- **Configuration**: LoRA (r=8, α=16), 200 samples, 1 epoch, lr `2e-7`, 50 optimizer steps
- **Training loss**: ~2.9 at the start, ~2.9 at the end (`train_loss = 2.866`), oscillating between 2.54 and 3.24

The learning rate `2e-7` is the script default, and the loss did not decrease meaningfully over 50 steps.

**Overall**: base 70.44% → fine-tuned 70.94% (**+0.50 pp, a difference of 1 question out of 203**)

At the item level the changes largely cancel out:

| Direction | Count | Question IDs |
|---|---|---|
| Wrong → correct | **5** | QA-000083, QA-000087, QA-000223, QA-000259, QA-000669 |
| Correct → wrong | **4** | QA-000251, QA-000455, QA-000700, QA-000726 |
| Net | **+1** | |

### Is this overfitting?

**There is no evidence of overfitting.** The four regressions are:

| ID | Subcategory | Gold | base → fine-tuned |
|---|---|---|---|
| QA-000251 | Geometric Relation | C | C → D |
| QA-000455 | Trend and Pattern | A | A → C |
| QA-000700 | Perspective and Viewpoint | A | A → D |

They are spread across unrelated subcategories rather than concentrated in one, and their number is
comparable to the improvements. Combined with the flat training loss (2.9 → 2.9), the most plausible
reading is that these flips are **decoding noise on essentially unchanged weights**, not overfitting.

One systematic change was observed: **output-format compliance** (`QA-000223`, Case 5 above).
Because all training targets share the `<answer>X</answer>` format, format-level signal is the first
thing a small amount of training would be expected to affect.

### What would be needed for a meaningful result

(a) raise the learning rate to roughly `1e-4`–`2e-4`; (b) scale the training set toward the full
15,773 converted samples; (c) train for more epochs; (d) match the frame count and resolution between
training and evaluation. As run, this experiment **validates that the pipeline works end to end**,
but it is not sufficient as a performance-improvement study.

## 5. Environment Issues and Code Changes

The provided infrastructure code assumes the library versions available when it was written. The current
environment (transformers 5.17.0, RTX 2080 Ti) required **seven fixes**. None of them are in
`TODO(student)` regions; all are in infrastructure files.

### 5.1 Library version drift (transformers 5.17.0)

| File | Problem | Fix |
|---|---|---|
| `reva_eval/inference_vllm_origin_number.py` | `AutoModelForVision2Seq` removed | Replaced with `AutoModelForImageTextToText` (2 sites) |
| `qwen_finetune/scripts/sft_7b.sh` | `warmup_ratio` removed from `TrainingArguments` | Dropped the argument |
| `qwen_finetune/qwenvl/train/trainer.py` | `apply_multimodal_rotary_pos_emb` removed | Made the import optional via `try/except` |

### 5.2 GPU architecture constraint (RTX 2080 Ti is Turing; FlashAttention 2 requires Ampere+)

Running unmodified produced `RuntimeError: FlashAttention only supports Ampere GPUs or newer`.

| File | Fix |
|---|---|
| `VILA/llava/model/multimodal_encoder/siglip_encoder.py` | `attn_implementation="flash_attention_2"` → `"sdpa"` (3 sites) |
| `qwen_finetune/qwenvl/train/train_qwen.py` | `train(attn_implementation="flash_attention_2")` → `"sdpa"` |
| `qwen_finetune/qwenvl/train/trainer.py` | Made the `flash_attn` import optional |
| `qwen_finetune/scripts/sft_7b.sh` | `--data_flatten True` → `False`, to avoid the flash-attn varlen kernel path |

### 5.3 VILA repository dependency conflicts

- `VILA/pyproject.toml`: the pin `timm==0.9.12` conflicts with `ps3-torch` (installed from GitHub HEAD),
  which requires `timm==1.0.15`. Relaxed to `timm>=0.9.12`.
- `VILA/llava/utils/media.py`: `getattr(config, "fps", 0.0)` returns an explicit `None` because the config
  class defines `fps=None`, causing `TypeError: '>' not supported between 'NoneType' and 'int'`.
  Changed to `getattr(config, "fps", 0.0) or 0.0`.

### 5.4 Memory constraints (11 GB VRAM)

| Symptom | Cause | Resolution |
|---|---|---|
| Evaluation always OOMs at exactly 32 questions | The dataloader batch size is **hard-coded to 8** | Introduced an `EVAL_BATCH_SIZE` environment variable; used 2 (base) and 1 (fine-tuned) |
| Training OOMs immediately | 4B bf16 weights (~8.7 GB) plus ~1.6 GB of resident usage leave no headroom | DeepSpeed ZeRO-3 with **CPU parameter offload** (new `scripts/zero3_offload.json`; added a `DEEPSPEED_CONFIG` environment variable) |
| The requested GPU was ignored | `eval_reva_v2.sh` overrides `CUDA_VISIBLE_DEVICES=$idx` (the chunk index) | Added an `EVAL_GPU` override |

**Resident memory differs per GPU** (GPU0 810 MB, GPU2 1.3 GB, GPU3 1.59 GB), so the same configuration
succeeded on one GPU and failed on another. This is why the fine-tuned evaluation failed on GPU3 with
batch size 2 but succeeded on GPU2 with batch size 1.

### 5.5 Other

- `ffmpeg` was not installed on the server; the static build shipped with `imageio-ffmpeg` was linked
  into `~/.local/bin/ffmpeg`.
- The server is shared. When all four GPUs were occupied by another user, jobs waited for capacity;
  the fine-tuned evaluation was later split into 4 parallel chunks once GPUs became free.

## 6. Deviations from the Recommended Settings

For reproducibility, the differences between the values recommended in `INSTRUCTIONS.md` and the values
actually used are listed below.

| Setting | Recommended | Used | Reason |
|---|---|---|---|
| Evaluation frames | `MAX_FRAMES=4` | `4` (same) | — |
| Evaluation size | Full test set | **50 videos / 203 questions** | Time budget |
| Evaluation batch size | 8 (hard-coded) | 2 (base) / 1 (fine-tuned) | 11 GB VRAM OOM |
| Training `MAX_PIXELS` | `50176` | **`28224`** | 11 GB VRAM OOM |
| Training `VIDEO_MAX_FRAMES` | `4` | **`2`** | 11 GB VRAM OOM |
| Training `MODEL_MAX_LENGTH` | `2048` | **`768`** | 11 GB VRAM OOM |
| Training parallelism | `NPROC_PER_NODE=2` + ZeRO-3 | **1 GPU + ZeRO-3 CPU offload** | Two free GPUs were rarely available on the shared server |
| Learning rate | `2e-7` (default) | `2e-7` (same) | — |

The reduced training settings **directly affect how the fine-tuning result should be read**: training used
2 frames at low resolution while evaluation used 4 frames at higher resolution, so the train/test input
conditions do not match. This is taken into account in the conclusions in §4.

## 7. Deliverables

| Item | Path |
|---|---|
| Model comparison | `outputs/model_comparison.csv` |
| Qwen base result | `outputs/qwen_base_real50e/qwen_base_real50e/result.csv` |
| Qwen fine-tuned result | `outputs/qwen_sft_real50_p4/qwen_sft_real50_p4/result.csv` |
| VILA result | `outputs/vila_reva_v2_real50/metrics.json` |
| Training checkpoint | `outputs/qwen_reva_sft_real/checkpoint-50/` |
| Completed code (GitHub) | https://github.com/shinnew9/CSE498_AI-Healthcare-Robotics/tree/main/Lab2_ReVA_VQA |

**Completed `TODO(student)` functions**

| File | Functions |
|---|---|
| `scripts/build_qwen_train_data.py` | `format_options`, `format_question`, `format_answer`, `iter_reva_qas`, `convert_item` |
| `reva_eval/data/rsvidqa/prepare_reva_v2_test_set.py` | `to_abs_path`, `to_rel_stem`, `get_qa_id`, `iter_flat_items` |
| `scripts/score_reva_predictions.py` | `extract_answer`, `extract_letter`, `load_predictions`, `score_predictions` |
| `vila_eval/reva_v2.py` | `load_instances`, `resolve_video_path`, `build_prompt`, `parse_choice`, `summarize` |

Unit tests (`pytest -q`): **7 passed**